# Blackboard presentation prototype — Colab T4

Run cells in order. First use **Runtime → Change runtime type → T4 GPU**.
Upload `outputs/blackboard_prototype_source.zip` from your local project when prompted.
This ZIP includes the current fixes; cloning the old GitHub `main` will not include them.

This is text-only real-model deliberation: a reasoner, critic and verifier share a board.
Answer agreement and grading are separate. Recovery is disabled. The five supplied
questions are curated prototype examples, not an official benchmark evaluation.

**First run one question.** Inspect its answer, trace, GPU check and runtime. Then change
`LIMIT = 3` below and rerun only the compute/viewer cells. Do not start 100–200 trials
before measuring the first run. Files are saved to Google Drive and can be downloaded.

In [ ]:
import sys, subprocess, json, os, time, shutil, hashlib, uuid, urllib.request
from pathlib import Path
from datetime import datetime, timezone
assert sys.version_info >= (3, 11), 'Python 3.11+ is required'
subprocess.run(['nvidia-smi'], check=True)

## Upload and verify the current source
On your laptop, the bundle is already prepared. If you edit the project later, regenerate
it with `python -m scripts.prototype_bundle`. Select just that source ZIP here.

In [ ]:
from google.colab import files
from zipfile import ZipFile
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one prototype source ZIP'
ROOT = Path('/content') / ('blackboard_prototype_' + uuid.uuid4().hex[:8])
ROOT.mkdir()
with ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (ROOT / member.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), 'Invalid path in ZIP'
    archive.extractall(ROOT)
bundle = json.loads((ROOT / 'prototype_source_manifest.json').read_text())
for name, expected in bundle['files'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
assert hashlib.sha256(json.dumps(bundle['files'], sort_keys=True).encode()).hexdigest() == bundle['source_digest']
print('Verified source:', bundle['source_digest'], 'base commit:', bundle['commit'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)], check=True)
os.chdir(ROOT)

## Install Ollama and configure one model
The installer comes from [official Ollama Linux instructions](https://docs.ollama.com/linux).
The selected model is `qwen2.5:7b-instruct-q4_K_M`, context 8192, sequential requests.
The runner will require a detected GPU **and** full model placement in GPU memory.

In [ ]:
MODEL = 'qwen2.5:7b-instruct-q4_K_M'
ENV = dict(os.environ)
ENV.update({
    'LLM_BACKEND': 'ollama', 'LLM_MODEL': MODEL,
    'OLLAMA_HOST': 'http://127.0.0.1:11434',
    'LLM_TEMPERATURE': '0.3', 'LLM_NUM_CTX': '8192',
    'LLM_MAX_TOKENS': '768', 'LLM_MAX_REPAIRS': '2',
    'LLM_TIMEOUT_S': '120', 'LLM_MAX_CONCURRENCY': '1', 'LLM_MAX_RETRIES': '2',
    'PYTHONPATH': str(ROOT / 'src'), 'MPLCONFIGDIR': '/content/blackboard_mpl',
    'OLLAMA_NUM_PARALLEL': '1', 'OLLAMA_MAX_LOADED_MODELS': '1',
})
if not shutil.which('ollama'):
    if not shutil.which('zstd'):
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
    installer = Path('/content/ollama_install.sh')
    urllib.request.urlretrieve('https://ollama.com/install.sh', installer)
    subprocess.run(['sh', str(installer)], check=True)

if 'OLLAMA_PROC' not in globals() or OLLAMA_PROC.poll() is not None:
    OLLAMA_LOG = Path('/content/blackboard_ollama.log')
    with OLLAMA_LOG.open('a') as log:
        OLLAMA_PROC = subprocess.Popen(['ollama', 'serve'], env=ENV, stdout=log, stderr=subprocess.STDOUT)
for attempt in range(60):
    try:
        with urllib.request.urlopen(ENV['OLLAMA_HOST'] + '/api/version', timeout=2) as response:
            print('Ollama:', json.load(response)); break
    except Exception:
        if OLLAMA_PROC.poll() is not None:
            raise RuntimeError(OLLAMA_LOG.read_text()[-5000:])
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not become ready; inspect ' + str(OLLAMA_LOG))
subprocess.run(['ollama', 'pull', MODEL], env=ENV, check=True)

## Persist results and check inference
Google Drive mounting needs your consent in Colab. Each run gets a fresh output directory.
The preflight report shows model availability, token reporting and GPU placement.
Do not proceed after a failed required check. The compute command adds `--require-gpu`.

In [ ]:
from google.colab import drive
# Use force_remount=False; rerunning does not create a second mount.
drive.mount('/content/drive')
OUT_ROOT = Path('/content/drive/MyDrive/blackboard_prototype')
OUT_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'scripts.s2_preflight', '--backend', 'ollama', '--model', MODEL],
               cwd=ROOT, env=ENV, check=True)

## Start compute in the background
Keep `LIMIT = 1` for the first run; this avoids blocking notebook controls and the viewer.
After it finishes, inspect the status cell. Change to `LIMIT = 3`, keep one seed, and
rerun this cell plus the viewer/status cells for the small presentation batch.
A running batch cannot be overwritten. Every question/seed has at most nine ordinary
turns, 100 backend attempts (including repairs/retries) and 20 minutes of inference.

For **natural disagreement and counterfactual intervention**, set `RECOVERY_DEMO = True`
and select `RECOVERY_TASK = 'biased_host'` (or `correlated_detectors`, `sampled_child`,
`two_envelopes`, `load_mix`). No initial debate is seeded. Recovery starts only
after a real conflict; unanimous ordinary agreement does not call the resolver.
Recovery peers calculate independently with the debate hidden before checking
the sandbox and live board. This adds up to six model calls; correctness is still graded separately.
Numerical posts also require exact calculator checks. Invalid arithmetic or stale
endorsements trigger repair; persistent errors fail the turn. The recovery policy
shown in the viewer should be `prototype-recovery-v8-grouping-and-ratify`.
See `docs/guides/prototype_recovery_demo.md` for trigger semantics and limitations.


In [ ]:
LIMIT = 1
SEEDS = [42]
RECOVERY_DEMO = False
RECOVERY_TASK = 'biased_host'
assert LIMIT in range(1, 6)
if 'COMPUTE_PROC' in globals() and COMPUTE_PROC.poll() is None:
    raise RuntimeError('A compute run is still active. Inspect it or stop it before starting another.')
if 'DEMO_PROC' in globals() and DEMO_PROC.poll() is None:
    DEMO_PROC.terminate(); DEMO_PROC.wait(timeout=10)
run_tag = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
RUN_DIR = OUT_ROOT / run_tag
COMPUTE_LOG = OUT_ROOT / (run_tag + '.log')
if RECOVERY_DEMO:
    assert len(SEEDS) == 1, 'Use one seed for this recovery demo'
    command = [sys.executable, '-u', '-m', 'scripts.prototype_recovery',
               '--backend', 'ollama', '--model', MODEL,
               '--tasks', 'examples/prototype/challenging_tasks.json',
               '--references', 'examples/prototype/challenging_references.json',
               '--task-id', RECOVERY_TASK, '--seed', str(SEEDS[0]),
               '--trigger', 'first-conflict', '--require-gpu', '--out', str(RUN_DIR)]
else:
    command = [sys.executable, '-u', '-m', 'scripts.prototype_run',
               '--backend', 'ollama', '--model', MODEL,
               '--tasks', 'examples/prototype/tasks.json',
               '--references', 'examples/prototype/references.json',
               '--seeds', *map(str, SEEDS), '--limit', str(LIMIT), '--max-turns', '9',
               '--max-calls', '100', '--timeout-s', '1200', '--require-gpu', '--out', str(RUN_DIR)]
with COMPUTE_LOG.open('w') as log:
    COMPUTE_PROC = subprocess.Popen(command, cwd=ROOT, env=ENV, stdout=log, stderr=subprocess.STDOUT)
for attempt in range(30):
    if (RUN_DIR / 'manifest.json').exists(): break
    if COMPUTE_PROC.poll() is not None:
        raise RuntimeError(COMPUTE_LOG.read_text()[-8000:])
    time.sleep(1)
else:
    raise RuntimeError('No run manifest; inspect ' + str(COMPUTE_LOG))
print('Compute PID:', COMPUTE_PROC.pid, '\nArtifacts:', RUN_DIR, '\nLog:', COMPUTE_LOG)

## Open the live artifact viewer
This shows real agent posts as compute writes them. Trial selection, replay and Follow
latest also work on completed runs. Colab's official iframe proxy keeps the server local;
there is no public tunnel. A first preflight/model load may take time before any posts appear.

In [ ]:
from google.colab import output
PORT = 8765
if 'DEMO_PROC' not in globals() or DEMO_PROC.poll() is not None:
    DEMO_LOG = Path('/content/blackboard_demo.log')
    with DEMO_LOG.open('w') as log:
        DEMO_PROC = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.prototype_demo',
                                     '--run-dir', str(RUN_DIR), '--port', str(PORT)],
                                    cwd=ROOT, env=ENV, stdout=log, stderr=subprocess.STDOUT)
for attempt in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/state', timeout=2) as response:
            json.load(response); break
    except Exception:
        if DEMO_PROC.poll() is not None:
            raise RuntimeError(DEMO_LOG.read_text()[-5000:])
        time.sleep(1)
else:
    raise RuntimeError('Viewer did not become ready; inspect ' + str(DEMO_LOG))
output.serve_kernel_port_as_iframe(PORT, height=850)

## Check progress and results (rerun whenever needed)
Exit status `0` means the run completed. `1` means preflight/internal/export failure;
`2` means at least one inference failure; `130` means interrupted. Turn/call/time limits
are valid recorded outcomes and can occur even with exit `0`; inspect the outcomes.
Answer agreement does not guarantee correctness. Exact grading may mark a verbose
otherwise sensible answer incorrect; keep the explicit answer format or review manually.

In [ ]:
status = COMPUTE_PROC.poll()
print('Compute:', 'RUNNING' if status is None else f'FINISHED (exit {status})')
print(COMPUTE_LOG.read_text()[-8000:])
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
print('Run status:', manifest['status'])
if manifest.get('error'): print('ERROR:', manifest['error'])
if manifest.get('export_error'): print('EXPORT ERROR:', manifest['export_error'])
if manifest.get('preflight'):
    for item in manifest['preflight']['checks']:
        print('PASS' if item['ok'] else 'FAIL/WARN', item['name'], item['detail'])
if (RUN_DIR / 'summary.json').exists():
    print(json.dumps(json.loads((RUN_DIR / 'summary.json').read_text()), indent=2))
    import pandas as pd
    display(pd.read_csv(RUN_DIR / 'summary.csv'))

## Download completed presentation artifacts
Wait for compute to finish first. Unzip on your laptop and open `demo.html` in a browser.
It contains the recorded trace and works without Colab, Ollama or internet. Use
`figures/cost_and_runtime.png`, `figures/outcomes.png`, and `summary.csv` in slides.
For a new live local viewer, run `python -m scripts.prototype_demo --run-dir <unzipped-folder>`
inside your installed local project and open `http://127.0.0.1:8765`.

In [ ]:
assert COMPUTE_PROC.poll() is not None, 'Wait for completion or stop compute before downloading'
assert (RUN_DIR / 'demo.html').exists(), 'No completed export; inspect the run log'
archive_path = shutil.make_archive('/content/blackboard_results_' + RUN_DIR.name, 'zip', RUN_DIR)
files.download(archive_path)
print('Drive copy:', RUN_DIR)

## Optional: stop an active run and keep written work
Run this cell only when you intend to cancel compute. SIGINT lets the runner save its
partial trial and exports. Keep the viewer/Ollama alive while inspecting the result.

In [ ]:
import signal
if 'COMPUTE_PROC' in globals() and COMPUTE_PROC.poll() is None:
    COMPUTE_PROC.send_signal(signal.SIGINT)
    try:
        COMPUTE_PROC.wait(timeout=30)
    except subprocess.TimeoutExpired:
        print('Still stopping. Wait and inspect the log; do not kill it while saving artifacts.')
print('Compute exit:', COMPUTE_PROC.poll() if 'COMPUTE_PROC' in globals() else 'not started')

After downloading artifacts, the runtime can be disconnected. Present the measured small
sample as a prototype update. Formal recovery, benchmark tool environments and broad
ablation claims are future work. [Full local instructions](../docs/guides/prototype_quickstart.md).